<a href="https://colab.research.google.com/github/hamza-6969/Twitter_sentimental_analysis/blob/main/Twitter_Sentiment_analysis_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install kaggle

In [5]:
# configuring the path of Kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [6]:
from zipfile import ZipFile
dataset  = '/content/archive.zip'

with ZipFile(dataset,"r") as zip :
  zip.extractall()
  print("data extracted")


data extracted


In [7]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

nltk.download("stopwords")
print(stopwords.words("english"))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [8]:
data = pd.read_csv("/content/training.1600000.processed.noemoticon.csv", encoding = "ISO-8859-1")

Data Processing and Cleaning

In [9]:
data.shape
data.head()
tweet_data = pd.read_csv("/content/training.1600000.processed.noemoticon.csv", names= ["target","ids","date","flag","user","data" ], encoding = "ISO-8859-1")

In [10]:
#replacing 4 by 1
tweet_data.replace({"target" : {4:1}}, inplace = True)
tweet_data["target"].value_counts()

,count
target,
0,800000
1,800000


In [11]:
#Stemming the data
port_stem = PorterStemmer()
def stemming(content) :
  stemmed_content = re.sub("[^a-zA-Z]"," ", content) #keeps only letters
  stemmed_content = stemmed_content.lower().split(" ") #splits it
  stemmed_content = [port_stem.stem(word) for word in stemmed_content if word not in stopwords.words("english")] # this stems words
  stemmed_content = " ".join(stemmed_content)

  return stemmed_content








In [12]:
# #applying it to df["data"]
tweet_data["stemmed_content"] = tweet_data["data"].apply(stemming)

In [13]:
X= tweet_data["stemmed_content"]
Y = tweet_data["target"]

X_train,X_test,Y_train,Y_test = train_test_split(X,Y, test_size = 0.2 , stratify = Y, random_state= 2)


#convert our textual tweet data into vectors(numbers) so that model understands

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)

model.fit(X_train,Y_train)






KeyError: 'stemmed_content'

In [ ]:
#Checking Accuracy of the model
X_prediction = model.predict(X_test)

score = accuracy_score(X_prediction,Y_test)
print(f"The model has a accuracy score of {score*100}%")

In [ ]:
import pickle

file_name = "trained_model.sav"

pickle.dump(model,open(file_name,"wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))


In [ ]:
loaded_model= pickle.load(open("/content/trained_model.sav","rb"))

check_tweet = X_test[:200]

for index,tweet in enumerate(check_tweet) :
  print(f"tweet number {index},prediction : {loaded_model.predict(tweet)}")
